In [ ]:
# --- Tutorial setup ---
# This cell loads everything needed to run this notebook standalone.
# It is hidden in the rendered documentation but visible when running the
# notebook as a tutorial.
import itertools
import os
import sys
sys.path.insert(0, "/".join(["..","python"]))

import matplotlib.pyplot as plt
from matplotlib import patches as mpl_patches
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
from gridr.core.convolution.fft_filtering import (
    fft_array_filter,
    BoundarySpec,
    ConvolutionMethod,
    OutputMode,
    DecimationOrigin,
)
from gridr.misc.mandrill import mandrill
from notebook_utils import plot_im, mpl_plot_wrapper
IN_DOC_BUILD = os.environ.get("DOC_BUILD", "0") == "1"
import notebook_utils
notebook_utils.FORCE_MPL_INTERACTIVE=True
if not IN_DOC_BUILD and not notebook_utils.FORCE_MPL_INTERACTIVE:
    from bokeh.io import output_notebook # enables plot interface in J notebook
    output_notebook()
from typing import get_args

mandrill_0 = mandrill[0]

@mpl_plot_wrapper
def plot_mesh3d_static(x, y, data, *, height=350, width=350, prefix=None):
    fig = plt.figure(figsize=(width / 100, height / 100), dpi=100)
    ax = fig.add_subplot(111, projection="3d") # Use original sizing strategy
    finite_data = data[np.isfinite(data)]
    
    if finite_data.size > 0:
        vmin, vmax = np.min(finite_data), np.max(finite_data)
    else:
        vmin, vmax = 0, 1
        
    X, Y = np.meshgrid(x, y)   
    ax.plot_wireframe(X, Y, data, rstride=1, cstride=1, color='blue', linewidth=0.3)
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    
    return fig
    
def gaussian_kernel_2d(shape, sigma):
    """
    Generates a normalized 2D Gaussian kernel.
    
    :param shape: Kernel size (int for a square kernel, or tuple (height, width))
    :param sigma: Standard deviation of the Gaussian (float or tuple (sigma_y, sigma_x))
    :return: a tuple containing :
        - 1D Numpy array representing x coordinates
        - 1D Numpy array representing y coordinates
        - 2D NumPy array representing the kernel
    """
    # Handle scalar or tuple inputs for shape and sigma
    if isinstance(shape, int):
        h = w = shape
    else:
        h, w = shape
        
    if isinstance(sigma, (int, float)):
        sigma_y = sigma_x = sigma
    else:
        sigma_y, sigma_x = sigma

    # Create centered coordinates (e.g., size 5 -> [-2, -1, 0, 1, 2])
    y = np.arange(-(h // 2), h // 2 + 1 if h % 2 != 0 else h // 2)
    x = np.arange(-(w // 2), w // 2 + 1 if w % 2 != 0 else w // 2)
    
    # 2D Grid
    x_grid, y_grid = np.meshgrid(x, y)
    
    # Calculate 2D Gaussian function
    kernel = np.exp(-0.5 * ((x_grid / sigma_x)**2 + (y_grid / sigma_y)**2))
    
    # Normalize so the sum of all elements equals 1
    kernel_sum = np.sum(kernel)
    if kernel_sum != 0:
        kernel /= kernel_sum
        
    return x, y, kernel

def generate_high_frequency_test_image(size=512, frequency_factor=150):
    """
    Generates a 2D zone plate (chirp) image containing high spatial frequencies.
    
    :param size: Image resolution (size x size)
    :param frequency_factor: Controls how rapidly the frequency increases towards the edges
    :return: 2D NumPy array representing the test image
    """
    x = np.linspace(-1, 1, size)
    y = np.linspace(-1, 1, size)
    X, Y = np.meshgrid(x, y)

    # Frequency increases proportionally to the squared distance from the center
    R2 = X**2 + Y**2
    return 0.5 + 0.5 * np.sin(frequency_factor * R2)
    
def _window_rect(window, **kwargs):
    """Build a matplotlib Rectangle from an inclusive window ((r0, r1), (c0, c1))."""
    (r0, r1), (c0, c1) = window
    return mpl_patches.Rectangle((c0 - 0.5, r0 - 0.5), c1 - c0 + 1, r1 - r0 + 1,
                     fill=False, **kwargs)


def valid_window_from_prod(window_prod, kernel_shape):
    """Compute the valid window (inclusive, full-image frame) from the same window.

    The kernel is assumed to be padded to an odd size: K_odd = K + (1 - K % 2).
    Returns None if the valid region is empty along any axis (N < K_odd).
    """
    window_prod = np.asarray(window_prod)
    k = np.asarray(kernel_shape[:2])
    radius = (k + (1 - k % 2)) // 2
    window_valid = window_prod + np.stack([radius, -radius], axis=1)
    if np.any(window_valid[:, 1] < window_valid[:, 0]):
        return None
    return window_valid


@mpl_plot_wrapper
def plot_image_output_mode(image_full, window_prod, kernel_shape, prefix=None,
                           scale=1, dpi=100):
    """Plot the full output with its same/valid regions at native resolution.

    Each image pixel is rendered as exactly scale x scale screen pixels
    (scale must be an integer), with no interpolation.
    """
    h, w = image_full.shape[:2]

    # Margins in screen pixels (integers to keep the axes aligned on the pixel grid)
    m_left, m_right, m_top, m_bottom = 10, 10, 24, 30
    ax_w, ax_h = w * scale, h * scale
    fig_w = ax_w + m_left + m_right
    fig_h = ax_h + m_top + m_bottom

    fig = plt.figure(figsize=(fig_w / dpi, fig_h / dpi), dpi=dpi)
    ax = fig.add_axes([m_left / fig_w, m_bottom / fig_h, ax_w / fig_w, ax_h / fig_h])
    ax.imshow(image_full, cmap="gray", interpolation="none", resample=False,
              aspect="equal")

    handles = [
        _window_rect(
            [[0, image_full.shape[0]-1], [0, image_full.shape[1]-1]],
            edgecolor="tab:red", linewidth=1, linestyle="-", label="full"
        ),
        _window_rect(
            np.asarray(window_prod), edgecolor="tab:blue",
            linewidth=1, label="same")
    ]
    window_valid = valid_window_from_prod(window_prod, kernel_shape)
    if window_valid is not None:
        handles.append(_window_rect(window_valid, edgecolor="tab:green", linewidth=1,
                                    linestyle="--", label="valid"))
    
    for patch in handles:
        ax.add_patch(patch)

    # imshow puts pixel centers on integers, so pixel edges sit at +/- 0.5
    ax.set_xlim(-0.5, w - 0.5)
    ax.set_ylim(h - 0.5, -0.5)
    ax.set_axis_off()

    win_h, win_w = np.asarray(window_prod)[:,1] - np.asarray(window_prod)[:,0] + 1
    fig.text(0.5, 1 - 4 / fig_h, f"full ({h}x{w}) - input ({win_h}×{win_w}) - kernel ({kernel_shape[0]}×{kernel_shape[1]})", ha="center", va="top", fontsize=8)
    fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0),
               ncol=len(handles), frameon=False, fontsize=7)
    return fig


# Getting started: your first filtering

This first tutorial presents the simplest call to ``fft_array_filter``, which computes the FFT-based convolution of an image with a user-defined kernel.

**What you'll learn**

- How to import and call ``fft_array_filter``
- How to define a normalized Gaussian kernel from its standard deviations
- What the ``FilterResult`` return type contains

## Setting things up

First we have to import the `fft_array_filter` method and associated types.

We also import the well known mandrill raster image.

In [ ]:
from gridr.core.convolution.fft_filtering import fft_array_filter
from gridr.misc.mandrill import mandrill

Please note the mandrill image has 3 color's channels. Here we limit the image to the first channel.

In [ ]:
mandrill_0 = mandrill[0]

The last element to define is the convolution kernel. We use a discrete normalized
Gaussian of size $(2r + 1) \times (2r + 1)$, given by:

$$
K(i, j) = \frac{1}{Z}
\exp\left( -\frac{1}{2} \left( \frac{i^2}{\sigma_y^2} + \frac{j^2}{\sigma_x^2} \right) \right),
\qquad -r \leq i, j \leq r
$$

where $i$ and $j$ are the row and column offsets from the kernel center, and $Z$ is the sum
of the unnormalized values, so that the kernel sums to 1.

In this example we set $\sigma_x = \sigma_y = 4$ and $r = 3\sigma = 12$, which gives a
25x25 kernel covering the Gaussian up to three standard deviations from its center.

In [ ]:
# The `gaussian_kernel_2d` computes the gaussian kernel - it implements the above mentionned formula.
gaussian_sigma = 4.
x, y, kernel = gaussian_kernel_2d(shape=int(gaussian_sigma*6+1), sigma=gaussian_sigma)

In [ ]:
_ = plot_mesh3d_static(x, y, kernel, prefix="001_gaussian_kernel_sigma4")

We call `fft_array_filter` with only the required parameters and keep the default values for the others. These optional parameters are described later in this guide.

In [ ]:
filtered_mandrill_0 = fft_array_filter(
        arr=mandrill_0,
        kernel=kernel,
    )

`fft_array_filter` returns a `FilterResult`, a simple `NamedTuple` with two fields:

- `data`: the filtered image;
- `window`: the position of the production window (the whole input image by default) in the coordinates of the full output. This window is discussed in more detail later.

In [ ]:
type(filtered_mandrill_0)

In [ ]:
type(filtered_mandrill_0.data), filtered_mandrill_0.data.shape

In [ ]:
plot_im({"input": mandrill_0, "output": filtered_mandrill_0.data}, prefix="001_filtered_mandrill_0")

You have now blurred your first image with GridR.
The edges of the filtered image appear darker than the rest of the image. This effect is explained in the next section.